# Pillar 1 — Data Exploration

Quick, reproducible profiling of the raw projects and employees datasets before transformation.

In [2]:
from pathlib import Path
import pandas as pd

repo_root = Path.cwd()
while not (repo_root / 'datasets').is_dir():
    if repo_root.parent == repo_root:
        raise RuntimeError('Run this notebook from inside the assessment repository.')
    repo_root = repo_root.parent

projects = pd.read_csv(repo_root / 'datasets' / 'projects.csv')
employees = pd.read_csv(repo_root / 'datasets' / 'employees.csv')

In [3]:
# Shape, types, and a representative sample
display(projects.shape, projects.dtypes, projects.head())
display(employees.shape, employees.dtypes, employees.head())

(500, 11)

project_id             object
project_name           object
department             object
status                 object
start_date             object
end_date               object
budget                float64
actual_cost           float64
project_manager_id     object
priority               object
region                 object
dtype: object

,project_id,project_name,department,status,start_date,end_date,budget,actual_cost,project_manager_id,priority,region
0,PRJ0001,MLOps Platform,Sustainability,Completed,2023-06-13,2023-12-01,1200000.0,907809.0,EMP0126,Medium,Dubai
1,PRJ0002,Tax Automation Platform - Q2,Marketing,In Progress,2025-02-28,NaN,800000.0,795986.0,EMP0045,Medium,Abu Dhabi
2,PRJ0003,ERP Consolidation Phase 2,Legal,Completed,2023-11-21,2024-12-16,80000.0,56981.0,EMP0405,Medium,Dubai
3,PRJ0004,Employee Wellness App - Q1,Marketing,Completed,2024-06-09,2025-08-17,800000.0,707314.0,EMP0270,Medium,Abu Dhabi
4,PRJ0005,Mobile Workforce App - Q4,Finance,In Progress,2023-04-25,NaN,2000000.0,1329426.0,EMP0478,High,Ras Al Khaimah


(1000, 12)

employee_id         object
full_name           object
email               object
department          object
role                object
level               object
hire_date           object
salary               int64
manager_id          object
region              object
status              object
years_experience     int64
dtype: object

,employee_id,full_name,email,department,role,level,hire_date,salary,manager_id,region,status,years_experience
0,EMP0001,Anjali Desai,anjali.desai@presight.ai,Operations,Senior Data Engineer,Mid,2019-08-06,17839,EMP0000,Abu Dhabi,Active,6
1,EMP0002,Mohammed Mansour,mohammed.mansour@presight.ai,Operations,Senior Data Engineer,Senior,2014-08-20,33928,EMP0000,Dubai,Active,11
2,EMP0003,Adam Rahman,adam.rahman@presight.ai,Legal,Compliance Analyst,Mid,2021-02-17,18763,EMP0000,Abu Dhabi,Active,4
3,EMP0004,Fatima Al Suwaidi,fatima.al.suwaidi@presight.ai,Procurement,Procurement Analyst,Mid,2024-05-30,20100,EMP0000,Dubai,Active,2
4,EMP0005,Priya Chen,priya.chen@presight.ai,Procurement,Procurement Analyst,Junior,2024-02-21,14370,EMP0000,Abu Dhabi,Active,0


In [4]:
# Projects: missing values and categorical distributions
display(projects.isna().sum().sort_values(ascending=False))
display(projects['status'].value_counts(dropna=False))
display(projects['priority'].value_counts(dropna=False))

end_date              286
start_date             65
actual_cost            60
budget                 32
project_id              0
project_name            0
department              0
status                  0
project_manager_id      0
priority                0
region                  0
dtype: int64

Completed      214
In Progress    189
Not Started     50
On Hold         47
Name: status, dtype: int64

Medium      213
High        154
Critical     68
Low          65
Name: priority, dtype: int64

In [5]:
# Employees: vectorised DQ detection counts
hire_date = pd.to_datetime(employees['hire_date'], errors='coerce')
missing_email = employees['email'].isna() | employees['email'].astype('string').str.strip().eq('')
invalid_hire_date = hire_date.isna() & employees['hire_date'].notna()
implausible_hire_date = hire_date.notna() & (
    hire_date.lt('1995-01-01') | hire_date.gt(pd.Timestamp.today())
)
negative_experience = pd.to_numeric(employees['years_experience'], errors='coerce').lt(0)
self_manager = employees['employee_id'].eq(employees['manager_id'])

pd.Series({
    'missing_email': missing_email.sum(),
    'invalid_hire_date': invalid_hire_date.sum(),
    'implausible_hire_date': implausible_hire_date.sum(),
    'negative_years_experience': negative_experience.sum(),
    'employee_is_own_manager': self_manager.sum(),
    'duplicate_employee_id': employees['employee_id'].duplicated().sum(),
})

missing_email                10
invalid_hire_date             8
implausible_hire_date         0
negative_years_experience     5
employee_is_own_manager       1
duplicate_employee_id         0
dtype: int64

In [6]:
# Salary distribution by level, used to investigate inconsistent level/salary records
employees.groupby('level')['salary'].describe()[['count', 'mean', 'min', '50%', 'max']]

,count,mean,min,50%,max
level,,,,,
Director,30.0,60434.800000,50464.0,60104.5,69078.0
Junior,187.0,15226.582888,12043.0,14252.0,76507.0
Lead,114.0,44016.210526,38067.0,44119.5,49976.0
Mid,366.0,20577.508197,17073.0,20639.5,23994.0
Senior,303.0,30534.323432,25124.0,30409.0,35948.0
